# Oversampling

### <i> Using SDV Framework

<br>
<br>

`SDV` learns the overall data distribution and generates synthetic records from that learned distribution. You have a wide variety of options (Synthesizer algorithms) you can work with when deciding how you want to create more data:

</i>
<br>

|Synthesizer|Approach|
|---|---|
|GaussianCopulaSynthesizer| statistical/correlation-based approach (REALLY robust)|
|CTGAN| neural network-based|
|TVAE| variational autoencoder approach (good for noise)|
|CopulaGAN| GAN + copula approach (handles huge ranges in values decently well)|


<i>



---

### TVAE

In [1]:
from sdv.metadata import Metadata
from sdv.single_table import TVAESynthesizer

import pandas as pd

# Load a new dataframe.
df = pd.read_csv("../../houses.csv")

df.drop(["id", "date", "lat", "long", 'zipcode'], axis=1, inplace=True)


# Create the metadata again
metadata = Metadata.detect_from_dataframe(
    data=df
)

# Place the metadata into the TVAE model approach
synthesizer = TVAESynthesizer(
    metadata
)

# Set and fit the data into our dataframe.
# TVAE takes more time than Copula,
# but considerably less than CTGAN (4-8 minutes)
synthesizer.fit(df)

# Make 5k rows using Gaussian Copula
synthetic_df = synthesizer.sample(
    num_rows=5000
)

c:\Users\tuomas.valtanen\dataeng_advda2026_lectures\DataEngineering_AdvDA_2026\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [2]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Let's make two histplots for each column. One for the Original dataset, the other for CTGAN.
# for column in df.columns:

#     # Create the canvas and create
#     fig, axes = plt.subplots(1, 2, figsize=(12, 5))
#     sns.histplot(df[column], ax=axes[0], color="red")
#     sns.histplot(synthetic_data[column], ax=axes[1], color="gray")

#     axes[0].set_title("Original")
#     axes[1].set_title("TVAE Synthetic")


<i>
Visually, this looks like the best fit for the dataset. The shapes of each distribution seems about right!

<br>

We visually checked and inspected each column's shape individually, but that's not enough. We need to check the fit for the pairs as well. This neat tool does both:

###  Using `SDV` evaluate Quality

In [3]:
# Let's use the evaluate_quality function in SDV's library
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    real_data=df,
    synthetic_data=synthetic_df,
    metadata=metadata
)

# The higher the percentile, the better the fit. 
quality_report

# Column shape: Compares real data vs synthetic data 

C:\Users\tuomas.valtanen\AppData\Local\Temp\ipykernel_14100\2975143818.py:2: FutureWarning: The evaluation functions are now accessible via the 'sdv.evaluation' module.
  from sdv.evaluation.single_table import evaluate_quality


Generating report ...

(1/4) Evaluating Column Shapes: |          | 0/16 [00:00<?, ?it/s]|

(1/4) Evaluating Column Shapes: |██████████| 16/16 [00:00<00:00, 107.15it/s]|
Column Shapes Score: 93.87%

(2/4) Evaluating Column Pair Trends: |██████████| 120/120 [00:00<00:00, 174.98it/s]|
Column Pair Trends Score: 93.25%

(3/4) Evaluating Cardinality: N/A
This property does not apply to single-table data.

(4/4) Evaluating Intertable Trends: N/A
This property does not apply to single-table data.

Overall Score (Average): 93.56%



---

### Studying and Comparing TVAE with the Original

<i> Never take synthetic data at face value, always inspect if it reflects reality first.

In [4]:
# switching to decimal notation for the variance checks
pd.set_option('display.float_format', lambda x: "%.2f" % x)

In [5]:
# We can compare the statistics between the two dataframes by using `.describe()`
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
count,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00
mean,540088.14,3.37,2.11,2079.90,15106.97,1.49,0.01,0.23,3.41,7.66,1788.39,291.51,1971.01,84.40,1986.55,12768.46
std,367127.20,0.93,0.77,918.44,41420.51,0.54,0.09,0.77,0.65,1.18,828.09,442.58,29.37,401.68,685.39,27304.18
min,75000.00,0.00,0.00,290.00,520.00,1.00,0.00,0.00,1.00,1.00,290.00,0.00,1900.00,0.00,399.00,651.00
25%,321950.00,3.00,1.75,1427.00,5040.00,1.00,0.00,0.00,3.00,7.00,1190.00,0.00,1951.00,0.00,1490.00,5100.00
50%,450000.00,3.00,2.25,1910.00,7618.00,1.50,0.00,0.00,3.00,7.00,1560.00,0.00,1975.00,0.00,1840.00,7620.00
75%,645000.00,4.00,2.50,2550.00,10688.00,2.00,0.00,0.00,4.00,8.00,2210.00,560.00,1997.00,0.00,2360.00,10083.00
max,7700000.00,33.00,8.00,13540.00,1651359.00,3.50,1.00,4.00,5.00,13.00,9410.00,4820.00,2015.00,2015.00,6210.00,871200.00


In [ ]:
# pretty close again!
synthetic_df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,540228.25,3.27,2.12,2144.11,10053.49,1.48,0.00,0.05,3.28,7.73,1867.01,272.85,1971.57,58.98,2010.21,9187.68
std,356342.46,0.67,0.79,817.72,10292.39,0.52,0.03,0.43,0.46,1.16,814.00,430.57,28.01,337.71,671.14,9090.42
min,79591.00,1.00,0.00,692.00,520.00,1.00,0.00,0.00,2.00,6.00,328.00,0.00,1900.00,0.00,800.00,651.00
25%,333341.75,3.00,1.75,1490.00,5188.00,1.00,0.00,0.00,3.00,7.00,1289.75,0.00,1952.00,0.00,1510.00,5536.50
50%,451487.00,3.00,2.25,1959.00,7604.50,1.00,0.00,0.00,3.00,7.00,1624.50,2.00,1977.00,0.00,1794.00,7393.50
75%,608949.75,4.00,2.50,2649.00,10844.00,2.00,0.00,0.00,4.00,8.00,2234.00,450.25,1997.00,0.00,2450.25,9765.25
max,3485973.00,5.00,4.77,5045.00,309718.00,3.00,1.00,4.00,5.00,12.00,6521.00,2942.00,2015.00,2015.00,5208.00,258184.00


In [7]:
df.var()

price           134782378397.25
bedrooms                   0.87
bathrooms                  0.59
sqft_living           843533.68
sqft_lot          1715658774.18
floors                     0.29
waterfront                 0.01
view                       0.59
condition                  0.42
grade                      1.38
sqft_above            685734.67
sqft_basement         195872.67
yr_built                 862.80
yr_renovated          161346.21
sqft_living15         469761.24
sqft_lot15         745518225.34
dtype: float64

In [ ]:
# the variances are off though, at some variables
synthetic_df.var()

price           126979946572.18
bedrooms                   0.45
bathrooms                  0.62
sqft_living           668661.67
sqft_lot           105933318.45
floors                     0.27
waterfront                 0.00
view                       0.18
condition                  0.21
grade                      1.33
sqft_above            662602.91
sqft_basement         185387.78
yr_built                 784.66
yr_renovated          114049.47
sqft_living15         450433.24
sqft_lot15          82635688.67
dtype: float64

In [9]:
synthetic_df

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
0,420753.00,4,2.50,2847,34168,2.00,0,0,3,9,2507,0,1990,0,2838,45736
1,246909.00,4,1.02,1781,8584,1.00,0,0,3,7,1433,0,1953,0,1589,9961
2,279578.00,2,1.00,978,2850,2.00,0,0,3,7,1260,3,2003,0,1523,4840
3,122232.00,2,1.00,1458,12746,1.00,0,0,4,6,1001,4,1922,0,1201,9390
4,437611.00,3,1.75,1623,11042,1.00,0,0,4,7,1349,393,1976,0,1960,9647
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,214370.00,3,1.00,1474,13206,1.00,0,0,3,7,1257,1,1981,0,1370,8113
4996,228056.00,2,1.00,1214,3645,1.00,0,0,3,7,1000,2,1988,0,1381,3937
4997,363993.00,4,2.50,2114,15990,2.00,0,0,3,8,2288,0,1986,0,2037,10551
4998,313623.00,3,1.75,1763,16865,1.00,0,0,4,7,1786,0,1966,0,1418,10897


In [10]:
# hmm, one weird occasion found with renovation year being 1 
# values like 1 => should be 0
synthetic_df.query("yr_renovated < yr_built and yr_renovated != 0")

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
59,434143.00,4,2.91,1853,3301,2.00,0,0,3,7,1680,1,2006,1,1526,5212
729,330115.00,3,2.00,1482,3262,2.00,0,0,3,7,1326,4,2004,2001,1384,4012
856,544481.00,3,1.75,1400,5112,1.00,0,0,3,7,1325,0,1962,1,1825,4800
1017,391495.00,3,2.00,1748,6682,2.00,0,0,3,7,1678,0,2003,1999,1547,5285
1250,495006.00,3,1.75,1355,10856,1.00,0,0,3,7,1408,393,1960,1,1646,9449
1418,510743.00,3,2.50,1952,5032,2.00,0,0,3,8,1473,527,2007,1,1635,3586
1712,286259.00,3,2.50,1282,3965,2.00,0,0,3,7,1290,130,1998,1,1514,4962
1830,2013054.00,5,3.14,4520,7431,2.00,0,0,3,11,4848,0,2006,2003,3359,7575
2306,506204.00,2,1.51,1636,520,3.00,0,0,3,7,827,2,2004,1999,906,1860
2471,701687.00,4,3.33,3216,4625,2.00,0,0,3,8,2318,659,2005,1,2445,5154
